# sft35: SFT pareado Qwen3-4B-Base vs Qwen3.5-4B-Base sobre los mismos puzzles NVARC (vocab 16)

`gate35b` (31-oct) midio el PRIOR de rejilla: empate (1,044). Este kernel mide lo que falta: si la generacion
3.5 **convierte el SFT de NVARC en habilidad de rejilla** mejor que la 3. Misma muestra de puzzles
(`sft35-data`), misma receta (vocab recortado a 16 tokens como NVARC, LoRA r=64 rsLoRA sobre todas las
lineales + embeddings/cabeza completas, lr 1e-4 coseno, ~9K secuencias <= 2048 tok, 1 epoca), dos
semillas de orden/aumentacion por brazo. 4 GPUs = 4 corridas independientes.

Medida (misma poblacion que gate35b, 24 baratas + 12 medias del eval publico, no vista por ningun brazo):
NLL/celda de la GT (canonica y mediana de 3 vistas), exactas greedy, pre y post SFT; curva de loss; tok/s.

**PRERREG**: la via 3.5 se REABRE si post-SFT ratio q35/q3 <= 0,85 en AMBAS semillas y gana >= 2/3
(signo p<0,05) con la banda semilla-a-semilla mas estrecha que el efecto. Empate => cerrada definitivamente.

In [ ]:
%%writefile /kaggle/working/_worker_src.py
import gc, json, os, sys, time, math, random
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()
ARM = os.environ["ARM"]; SEED = int(os.environ["SEED"]); SHARD = int(os.environ["SHARD"])
N_SEQ = int(os.environ.get("N_SEQ", "9000")); MAX_LEN = int(os.environ.get("MAX_LEN", "2048"))
TRAIN_SECONDS = float(os.environ.get("TRAIN_SECONDS", "10800"))
ACCUM = int(os.environ.get("ACCUM", "8")); LR = float(os.environ.get("LR", "1e-4"))
LORA_R = int(os.environ.get("LORA_R", "64")); LORA_ALPHA = int(os.environ.get("LORA_ALPHA", "32"))
N_BARATAS = 24; N_MEDIAS = 12
SUCIAS = set(json.loads(os.environ["SUCIAS"]))
TAG = "%s_s%d" % (ARM, SEED)
OUTDIR = "/kaggle/working/%s" % TAG; os.makedirs(OUTDIR, exist_ok=True)
def log(*a):
    print("[%s %5.0fs]" % (TAG, time.time() - T00), *a, flush=True)

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import transformers
from transformers.utils.import_utils import is_flash_linear_attention_available
log("transformers", transformers.__version__, "FLA", is_flash_linear_attention_available(), "torch", torch.__version__)

# ---------------- datos ----------------
def buscar(nombre):
    for root, dirs, files in os.walk("/kaggle/input"):
        if nombre in files:
            return os.path.join(root, nombre)
    return None
def buscar_modelo(*marcas):
    mejor = None
    for root, dirs, files in os.walk("/kaggle/input"):
        low = root.lower().replace("\\", "/")
        if "config.json" in files and all(m in low for m in marcas):
            if any(x.endswith(".safetensors") for x in files): return root
            mejor = mejor or root
    return mejor
MDIR = buscar_modelo("qwen-3/", "/4b-base") if ARM == "q3" else buscar_modelo("q35_4b_base")
F_MUESTRA = buscar("nvarc_muestra.json")
F_CHAL = buscar("arc-agi_evaluation_challenges.json")
assert MDIR and F_MUESTRA and F_CHAL, (MDIR, F_MUESTRA, F_CHAL)
log("modelo", MDIR); log("datos", F_MUESTRA)
PUZZLES = json.load(open(F_MUESTRA))
CHAL = json.load(open(F_CHAL)); SOLS = json.load(open(os.path.join(os.path.dirname(F_CHAL), "arc-agi_evaluation_solutions.json")))
def coste(t):
    return sum(len(g["input"])*len(g["input"][0]) + len(g["output"])*len(g["output"][0]) for g in t["train"])
ORD = sorted([k for k in CHAL if k not in SUCIAS], key=lambda k: coste(CHAL[k]))
n = len(ORD); medio = ORD[int(0.40*n):int(0.60*n)]
KEYS_EVAL = ORD[:N_BARATAS] + medio[::max(1, len(medio)//N_MEDIAS)][:N_MEDIAS]
ESTRATO = {k: ("barata" if k in ORD[:N_BARATAS] else "media") for k in KEYS_EVAL}
log("eval:", len(KEYS_EVAL), "tareas; puzzles:", len(PUZZLES))

# ---------------- vocabulario de 16 tokens (como NVARC) ----------------
# 0-9 digitos, 10 '\n', 11 user, 12 assistant, 13 <|endoftext|>, 14 <|im_start|>, 15 <|im_end|>
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MDIR, trust_remote_code=True)
def uno(s):
    ids = tok.encode(s, add_special_tokens=False); assert len(ids) == 1, (s, ids); return ids[0]
IDS16 = [uno(str(d)) for d in range(10)] + [uno("\n"), uno("user"), uno("assistant"),
         tok.convert_tokens_to_ids("<|endoftext|>"), tok.convert_tokens_to_ids("<|im_start|>"), tok.convert_tokens_to_ids("<|im_end|>")]
assert all(isinstance(i, int) and i >= 0 for i in IDS16), IDS16
log("IDS16", IDS16)
NL, USER, ASSIST, PAD, IMS, IME = 10, 11, 12, 13, 14, 15

def grid_ids(g):
    out = []
    for i, fila in enumerate(g):
        if i: out.append(NL)
        out.extend(int(c) for c in fila)
    return out
def par_ids(inp, out, con_salida=True):
    """ids y mascara (1 = token entrenable/evaluable) de un par; formato de produccion."""
    ids = [IMS, USER, NL] + grid_ids(inp) + [IME, IMS, ASSIST, NL]
    m = [0] * len(ids)
    if con_salida:
        o = grid_ids(out) + [IME]
        ids += o; m += [1] * len(o)
    return ids, m
def seq_ids(pares, con_salida_final=True):
    ids, m = [], []
    for j, p in enumerate(pares):
        a, b = par_ids(p["input"], p["output"], con_salida=(con_salida_final or j < len(pares) - 1))
        ids += a; m += b
    return ids, m
def len_par(p):
    return 8 + sum(len(p[k]) * len(p[k][0]) + len(p[k]) - 1 for k in ("input", "output"))

# ---------------- aumentacion (D4 x permutacion de 10 colores) ----------------
def d4(g, k):
    a = np.asarray(g)
    if k >= 4: a = a.T
    return np.rot90(a, k % 4)
def aug(pares, rng):
    k = rng.randrange(8); perm = list(range(10)); rng.shuffle(perm); perm = np.asarray(perm)
    return [{"input": perm[d4(p["input"], k)].tolist(), "output": perm[d4(p["output"], k)].tolist()} for p in pares]

def construir_secuencias(n_seq, seed):
    rng = random.Random(1000 + seed)
    seqs = []; i = 0; vueltas = 0
    while len(seqs) < n_seq and vueltas < 3:
        if i >= len(PUZZLES): i = 0; vueltas += 1
        pares = list(PUZZLES[i]["pares"]); i += 1
        rng.shuffle(pares)
        pares = aug(pares, rng)
        while len(pares) >= 2 and sum(len_par(p) for p in pares) > MAX_LEN: pares.pop(0)
        if len(pares) < 2: continue
        ids, m = seq_ids(pares)
        if len(ids) > MAX_LEN: continue
        seqs.append((ids, m))
    return seqs
SEQS = construir_secuencias(N_SEQ, SEED)
n_tok = sum(len(s[0]) for s in SEQS); n_lab = sum(sum(s[1]) for s in SEQS)
log("secuencias", len(SEQS), "tokens", n_tok, "etiquetas", n_lab, "media", round(n_tok / max(1, len(SEQS))))

# ---------------- modelo ----------------
CLASES = ("AutoModelForImageTextToText", "AutoModelForCausalLM", "AutoModelForVision2Seq", "AutoModel")
def cargar(dir_):
    kw = dict(trust_remote_code=True, low_cpu_mem_usage=True, dtype=torch.bfloat16, device_map={"": 0}, output_loading_info=True)
    for nombre in CLASES:
        try:
            cls = getattr(transformers, nombre)
            try:
                m, info = cls.from_pretrained(dir_, **kw)
            except TypeError:
                kw2 = dict(kw); kw2["torch_dtype"] = kw2.pop("dtype"); m, info = cls.from_pretrained(dir_, **kw2)
        except Exception as e:
            log("  ", nombre, "error", str(e)[:160]); continue
        falt = list(info.get("missing_keys") or [])
        log("  ", nombre, "faltan", len(falt), "inesperadas", len(info.get("unexpected_keys") or []), "params", sum(p.numel() for p in m.parameters()) // 10**6, "M")
        if len(falt) == 0: return m, nombre
        del m; gc.collect(); torch.cuda.empty_cache()
    raise RuntimeError("sin clase que cargue " + dir_)
model, clase = cargar(MDIR)
model.config.use_cache = False

# recorte de vocabulario: embeddings y cabeza a 16 filas (entrenables completas, no LoRA)
H = model.get_input_embeddings().weight.shape[1]
emb_old = model.get_input_embeddings().weight.data
head_old = model.get_output_embeddings().weight.data
tied = bool(emb_old.data_ptr() == head_old.data_ptr())
emb = nn.Embedding(16, H, dtype=torch.bfloat16, device=emb_old.device)
emb.weight.data.copy_(emb_old[IDS16])
head = nn.Linear(H, 16, bias=False, dtype=torch.bfloat16, device=emb_old.device)
head.weight.data.copy_(head_old[IDS16])
if tied: head.weight = emb.weight
model.set_input_embeddings(emb); model.set_output_embeddings(head)
for c in (model.config, getattr(model.config, "text_config", None)):
    if c is not None:
        try: c.vocab_size = 16
        except Exception: pass
if hasattr(model, "vocab_size"): model.vocab_size = 16
log("vocab recortado a 16; H =", H, "atados =", tied, "clase =", clase)
del emb_old, head_old; gc.collect(); torch.cuda.empty_cache()

# LoRA sobre TODAS las lineales del decodificador (trato igual a ambas arquitecturas)
from peft import LoraConfig, get_peft_model
lineales = sorted({n.split(".")[-1] for n, mod in model.named_modules() if isinstance(mod, nn.Linear) and mod is not head and "visual" not in n and "vision" not in n})
log("modulos lineales objetivo:", lineales)
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias="none", use_rslora=True, target_modules=lineales))
for p_ in list(emb.parameters()) + list(head.parameters()): p_.requires_grad_(True)
params = [p for p in model.parameters() if p.requires_grad]
n_tr = sum(p.numel() for p in params)
log("parametros entrenables: %.1fM" % (n_tr / 1e6))
# v3: checkpointing de gradiente DESDE EL INICIO en todos los brazos. En v2 los brazos q3 saltaron a
# checkpointing tras el primer OOM (torch) y los q35 murieron con un OOM de Triton (kernels FLA en el
# backward) que no es torch.cuda.OutOfMemoryError. Mismo regimen para ambos = comparacion justa.
ck = bool(int(os.environ.get("CKPT", "1")))
if ck:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    log("gradient checkpointing activo desde el inicio")
model.train()

def forward_logits(ids):
    out = model(input_ids=ids, use_cache=False, return_dict=True)
    return out.logits

def nll_eval(ids, m):
    """suma de NLL sobre los tokens con mascara 1 y numero de celdas (digitos)."""
    with torch.no_grad():
        t = torch.tensor([ids], device="cuda")
        lg = forward_logits(t)[0, :-1].float()
        lp = lg.log_softmax(-1).gather(-1, t[0, 1:].unsqueeze(-1)).squeeze(-1)
        mm = torch.tensor(m[1:], device="cuda", dtype=torch.bool)
        tgt = t[0, 1:]
        celdas = int(((tgt < 10) & mm).sum().item())
        return float(-lp[mm].sum().item()), celdas

def greedy(ids, max_new):
    with torch.no_grad():
        t = torch.tensor([ids], device="cuda")
        out = model(input_ids=t, use_cache=True, return_dict=True)
        pkv = out.past_key_values; nxt = int(out.logits[0, -1].argmax()); gen = [nxt]
        while nxt != IME and len(gen) < max_new:
            out = model(input_ids=torch.tensor([[nxt]], device="cuda"), past_key_values=pkv, use_cache=True, return_dict=True)
            pkv = out.past_key_values; nxt = int(out.logits[0, -1].argmax()); gen.append(nxt)
        return gen
def ids_a_grid(gen):
    filas, fila = [], []
    for x in gen:
        if x == IME: break
        if x == NL: filas.append(fila); fila = []
        elif x < 10: fila.append(x)
        else: return None
    if fila: filas.append(fila)
    if not filas or any(len(f) != len(filas[0]) for f in filas) or len(filas[0]) == 0: return None
    return filas

VISTAS = [(0, None), (1, [0, 5, 3, 8, 1, 9, 2, 7, 4, 6]), (6, [9, 0, 4, 1, 7, 2, 8, 3, 5, 6])]
def vista(pares, k, perm):
    pp = np.asarray(perm) if perm else None
    f = lambda g: (pp[d4(g, k)] if pp is not None else d4(g, k)).tolist()
    return [{"input": f(p["input"]), "output": f(p["output"])} for p in pares]

def evaluar(etiqueta):
    model.eval(); filas = []; t0 = time.time()
    for key in KEYS_EVAL:
        task = CHAL[key]; gt = SOLS[key][0]
        pares = task["train"] + [{"input": task["test"][0]["input"], "output": gt}]
        fila = {"tarea": key, "estrato": ESTRATO[key]}
        nlls = []
        for vi, (k, perm) in enumerate(VISTAS):
            pv = vista(pares, k, perm)
            ids, m = seq_ids(pv)
            if len(ids) > 8192: fila["largo"] = len(ids); break
            s, c = nll_eval(ids, m)
            nlls.append(s / max(1, c))
            if vi == 0:
                fila["celdas"] = c; fila["n_tok"] = len(ids)
                ids_q, _ = seq_ids(pv, con_salida_final=False)
                try:
                    g = ids_a_grid(greedy(ids_q, 31 * 30 + 2))
                except Exception as e:
                    log("greedy fallo en", key, str(e)[:120]); g = None; fila["greedy_error"] = str(e)[:120]
                fila["exacta"] = bool(g == pv[-1]["output"])
        if nlls:
            fila["nll_celda"] = nlls[0]; fila["nll_celda_med3"] = float(np.median(nlls)); fila["nll_vistas"] = nlls
        filas.append(fila)
    model.train()
    ex = sum(1 for f in filas if f.get("exacta")); v = [f["nll_celda"] for f in filas if "nll_celda" in f]
    log("EVAL %s: exactas %d/%d | NLL/celda mediana %.3f media %.3f | %.0fs" % (etiqueta, ex, len(filas), float(np.median(v)), float(np.mean(v)), time.time() - t0))
    return filas

res = {"arm": ARM, "seed": SEED, "modelo": MDIR, "clase": clase, "ids16": IDS16, "tied": tied, "H": H,
       "n_seq": len(SEQS), "n_tok": n_tok, "n_lab": n_lab, "params_tr_M": round(n_tr / 1e6, 1), "lineales": lineales,
       "cfg": {"lr": LR, "accum": ACCUM, "r": LORA_R, "alpha": LORA_ALPHA, "max_len": MAX_LEN, "train_seconds": TRAIN_SECONDS},
       "fla": bool(is_flash_linear_attention_available()), "transformers": transformers.__version__}
res["eval_pre"] = evaluar("pre-SFT")

# ---------------- entrenamiento ----------------
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, betas=(0.9, 0.99))
pasos_plan = max(1, len(SEQS) // ACCUM); warm = max(1, int(0.03 * pasos_plan))
def lr_en(paso):
    if paso < warm: return LR * (paso + 1) / warm
    pr = (paso - warm) / max(1, pasos_plan - warm)
    return LR * 0.5 * (1 + math.cos(math.pi * min(1.0, pr)))
curva = []; t_tr = time.time(); tok_vistos = 0; lab_vistos = 0; paso = 0; acc_loss = 0.0; acc_n = 0
i = 0
while i < len(SEQS):
    if time.time() - t_tr > TRAIN_SECONDS:
        log("tope de tiempo de entrenamiento alcanzado en la secuencia", i); break
    ids, m = SEQS[i]
    t = torch.tensor([ids], device="cuda"); mm = torch.tensor(m[1:], device="cuda", dtype=torch.bool)
    try:
        lg = forward_logits(t)[0, :-1].float()
        loss = F.cross_entropy(lg[mm], t[0, 1:][mm])
        (loss / ACCUM).backward()
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if not ("out of memory" in str(e).lower()) or ck: raise
        log("OOM: activo gradient checkpointing y reintento"); ck = True
        opt.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        continue
    acc_loss += float(loss.item()); acc_n += 1; tok_vistos += len(ids); lab_vistos += int(mm.sum().item()); i += 1
    if i % ACCUM == 0 or i == len(SEQS):
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        for g in opt.param_groups: g["lr"] = lr_en(paso)
        opt.step(); opt.zero_grad(set_to_none=True); paso += 1
        if paso % 25 == 0 or paso == 1:
            el = time.time() - t_tr
            curva.append({"paso": paso, "seq": i, "tok": tok_vistos, "loss": acc_loss / max(1, acc_n), "lr": lr_en(paso), "s": round(el)})
            log("paso %d seq %d tok %d loss %.4f lr %.2e | %.0f tok/s | mem %.1f GB" % (paso, i, tok_vistos, acc_loss / max(1, acc_n), lr_en(paso), tok_vistos / el, torch.cuda.max_memory_allocated() / 1e9))
            acc_loss = 0.0; acc_n = 0
el = time.time() - t_tr
res.update({"train": {"seq_hechas": i, "pasos": paso, "tok": tok_vistos, "lab": lab_vistos, "s": round(el), "tok_s": round(tok_vistos / max(1, el)),
                      "ckpt": ck, "mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 1), "curva": curva}})
log("entrenamiento: %d seq, %d pasos, %d tok, %.0f s, %.0f tok/s" % (i, paso, tok_vistos, el, tok_vistos / max(1, el)))

# ---------------- evaluacion post y guardado ----------------
res["eval_post"] = evaluar("post-SFT")
# v3 murio AQUI en los 4 brazos q35: get_peft_model_state_dict lee config.vocab_size y Qwen3_5Config no lo
# expone a nivel raiz (vive en text_config). Guardar res.json ANTES del adapter y blindar el atributo.
json.dump(res, open(os.path.join(OUTDIR, "res.json"), "w"))
try:
    if not hasattr(model.config, "vocab_size"): model.config.vocab_size = 16
except Exception: pass
from peft import get_peft_model_state_dict
sd = {k: v.detach().cpu() for k, v in get_peft_model_state_dict(model).items()}
sd["__emb16__"] = emb.weight.detach().cpu(); sd["__head16__"] = head.weight.detach().cpu()
torch.save(sd, os.path.join(OUTDIR, "adapter.pt"))
json.dump(res, open(os.path.join(OUTDIR, "res.json"), "w"))
log("guardado", OUTDIR, "wall total %.0f s" % (time.time() - T00))


In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
SUCIAS = ["0934a4d8", "136b0064", "16b78196", "981571dc", "aa4ec2a5", "da515329"]

def hallar_wheels(patron):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.startswith(patron) and f.endswith(".whl") for f in files):
            return root
    return None
W_TF = hallar_wheels("transformers-"); W_FLA = hallar_wheels("flash_linear_attention")
print("wheels transformers:", W_TF, "| wheels fla:", W_FLA, flush=True)
if W_TF:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--upgrade", "--find-links", W_TF,
                        "transformers", "tokenizers", "safetensors", "huggingface_hub", "accelerate", "regex"], capture_output=True, text=True)
    print("pip tf rc=%d" % r.returncode, (r.stdout + r.stderr)[-500:], flush=True)
if W_FLA:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--find-links", W_FLA,
                        "flash-linear-attention", "fla-core", "einops"], capture_output=True, text=True)
    print("pip fla rc=%d" % r.returncode, (r.stdout + r.stderr)[-500:], flush=True)
# peft 0.20 aborta la inyeccion de LoRA si ve torchao < 0.16 (la imagen trae 0.10): sin torchao, peft lo ignora
r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True, text=True)
print("pip uninstall torchao rc=%d" % r.returncode, (r.stdout + r.stderr)[-200:], flush=True)
r = subprocess.run([sys.executable, "-c", "import transformers, peft;from transformers.utils.import_utils import is_flash_linear_attention_available as f;print(transformers.__version__, 'peft', peft.__version__, 'FLA', f())"], capture_output=True, text=True)
print("entorno:", (r.stdout + r.stderr).strip()[-400:], flush=True)

# v3: los brazos q3 (s1, s2) ya estan medidos en v2 (sft35.json v2); ahora solo q35, 4 semillas
BRAZOS = json.loads(os.environ.get("BRAZOS", '[["q35", 1], ["q35", 2], ["q35", 3], ["q35", 4]]'))
procs = []
for i, (arm, seed) in enumerate(BRAZOS):
    env = dict(os.environ)
    env.update({"CUDA_VISIBLE_DEVICES": str(i), "SHARD": str(i), "ARM": arm, "SEED": str(seed), "SUCIAS": json.dumps(SUCIAS),
                "N_SEQ": os.environ.get("N_SEQ", "9000"), "TRAIN_SECONDS": os.environ.get("TRAIN_SECONDS", "10800")})
    fh = open("/kaggle/working/log_%s_s%d.txt" % (arm, seed), "w")
    procs.append((arm, seed, subprocess.Popen([sys.executable, "/kaggle/working/_worker_src.py"], env=env, stdout=fh, stderr=subprocess.STDOUT), fh))
for arm, seed, p, fh in procs:
    p.wait(); fh.close()
    print("worker %s_s%d rc=%d a los %.0fs" % (arm, seed, p.returncode, time.time() - T0), flush=True)

R = json.loads("{\"q3_s1\": {\"arm\": \"q3\", \"seed\": 1, \"modelo\": \"/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1\", \"clase\": \"AutoModelForCausalLM\", \"ids16\": [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 198, 872, 77091, 151643, 151644, 151645], \"tied\": true, \"H\": 2560, \"n_seq\": 9000, \"n_tok\": 15509094, \"n_lab\": 6078732, \"params_tr_M\": 132.2, \"lineales\": [\"down_proj\", \"gate_proj\", \"k_proj\", \"o_proj\", \"q_proj\", \"up_proj\", \"v_proj\"], \"cfg\": {\"lr\": 0.0001, \"accum\": 8, \"r\": 64, \"alpha\": 32, \"max_len\": 2048, \"train_seconds\": 10800.0}, \"fla\": true, \"transformers\": \"5.16.1\", \"eval_pre\": [{\"tarea\": \"e8686506\", \"estrato\": \"barata\", \"celdas\": 105, \"n_tok\": 748, \"exacta\": false, \"nll_celda\": 1.4696684337797619, \"nll_celda_med3\": 1.3998081752232143, \"nll_vistas\": [1.4696684337797619, 1.3540909540085566, 1.3998081752232143]}, {\"tarea\": \"20270e3b\", \"estrato\": \"barata\", \"celdas\": 226, \"n_tok\": 709, \"exacta\": false, \"nll_celda\": 0.6027137283730296, \"nll_celda_med3\": 0.6064603687387652, \"nll_vistas\": [0.6027137283730296, 0.6064603687387652, 0.6350239441458103]}, {\"tarea\": \"7b5033c1\", \"estrato\": \"barata\", \"celdas\": 71, \"n_tok\": 976, \"exacta\": false, \"nll_celda\": 1.5271193813270247, \"nll_celda_med3\": 1.6248020655672315, \"nll_vistas\": [1.5271193813270247, 1.672822119484485, 1.6248020655672315]}, {\"tarea\": \"b0039139\", \"estrato\": \"barata\", \"celdas\": 212, \"n_tok\": 957, \"exacta\": false, \"nll_celda\": 0.7913860824872863, \"nll_celda_med3\": 0.7913860824872863, \"nll_vistas\": [0.7913860824872863, 0.7839133784456073, 0.8038517214217276]}, {\"tarea\": \"78332cb0\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 930, \"exacta\": false, \"nll_celda\": 0.3866337585449219, \"nll_celda_med3\": 0.4537655258178711, \"nll_vistas\": [0.3866337585449219, 0.4537655258178711, 0.5057917785644531]}, {\"tarea\": \"28a6681f\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.2178778839111328, \"nll_celda_med3\": 0.22701530456542968, \"nll_vistas\": [0.2178778839111328, 0.2667096710205078, 0.22701530456542968]}, {\"tarea\": \"d35bdbdc\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.3074209213256836, \"nll_celda_med3\": 0.3194326972961426, \"nll_vistas\": [0.3074209213256836, 0.3194326972961426, 0.35251895904541014]}, {\"tarea\": \"dbff022c\", \"estrato\": \"barata\", \"celdas\": 473, \"n_tok\": 1048, \"exacta\": false, \"nll_celda\": 0.18631502437793176, \"nll_celda_med3\": 0.20239751384323798, \"nll_vistas\": [0.18631502437793176, 0.20239751384323798, 0.21213220189036072]}, {\"tarea\": \"4c3d4a41\", \"estrato\": \"barata\", \"celdas\": 480, \"n_tok\": 1026, \"exacta\": false, \"nll_celda\": 0.17859843571980794, \"nll_celda_med3\": 0.18151445388793946, \"nll_vistas\": [0.17859843571980794, 0.18151445388793946, 0.19257771174112956]}, {\"tarea\": \"446ef5d2\", \"estrato\": \"barata\", \"celdas\": 1016, \"n_tok\": 2146, \"exacta\": false, \"nll_celda\": 0.15325765534648744, \"nll_celda_med3\": 0.1737351304902805, \"nll_vistas\": [0.15325765534648744, 0.18151092529296875, 0.1737351304902805]}, {\"tarea\": \"dd6b8c4b\", \"estrato\": \"barata\", \"celdas\": 484, \"n_tok\": 1080, \"exacta\": false, \"nll_celda\": 0.19546146235190148, \"nll_celda_med3\": 0.19546146235190148, \"nll_vistas\": [0.19546146235190148, 0.19415380935038418, 0.22238597396976692]}, {\"tarea\": \"3dc255db\", \"estrato\": \"barata\", \"celdas\": 544, \"n_tok\": 1204, \"exacta\": false, \"nll_celda\": 0.27532117507036996, \"nll_celda_med3\": 0.27955470365636487, \"nll_vistas\": [0.27532117507036996, 0.27955470365636487, 0.29760585111730237]}, {\"tarea\": \"16de56c4\", \"estrato\": \"barata\", \"celdas\": 582, \"n_tok\": 1262, \"exacta\": false, \"nll_celda\": 0.34055152709541453, \"nll_celda_med3\": 0.3683308080299613, \"nll_vistas\": [0.34055152709541453, 0.36843384418290914, 0.3683308080299613]}, {\"tarea\": \"97d7923e\", \"estrato\": \"barata\", \"celdas\": 600, \"n_tok\": 1304, \"exacta\": false, \"nll_celda\": 0.09632901509602865, \"nll_celda_med3\": 0.09985081354777019, \"nll_vistas\": [0.09632901509602865, 0.09985081354777019, 0.12942376454671223]}, {\"tarea\": \"2b83f449\", \"estrato\": \"barata\", \"celdas\": 746, \"n_tok\": 1596, \"exacta\": false, \"nll_celda\": 0.4448564135676734, \"nll_celda_med3\": 0.4448564135676734, \"nll_vistas\": [0.4448564135676734, 0.4003213263708529, 0.4914749186416094]}, {\"tarea\": \"bf45cf4b\", \"estrato\": \"barata\", \"celdas\": 985, \"n_tok\": 1791, \"exacta\": false, \"nll_celda\": 0.22940069673025063, \"nll_celda_med3\": 0.21452362525281568, \"nll_vistas\": [0.22940069673025063, 0.21452362525281568, 0.20762519642786326]}, {\"tarea\": \"faa9f03d\", \"estrato\": \"barata\", \"celdas\": 832, \"n_tok\": 1800, \"exacta\": false, \"nll_celda\": 0.19694962868323693, \"nll_celda_med3\": 0.20706682938795823, \"nll_vistas\": [0.19694962868323693, 0.20706682938795823, 0.23652780972994292]}, {\"tarea\": \"eee78d87\", \"estrato\": \"barata\", \"celdas\": 1024, \"n_tok\": 1280, \"exacta\": false, \"nll_celda\": 0.13134634494781494, \"nll_celda_med3\": 0.14614208042621613, \"nll_vistas\": [0.13134634494781494, 0.14614208042621613, 0.15498989820480347]}, {\"tarea\": \"31f7f899\", \"estrato\": \"barata\", \"celdas\": 732, \"n_tok\": 1592, \"exacta\": false, \"nll_celda\": 0.15093873237651553, \"nll_celda_med3\": 0.15093873237651553, \"nll_vistas\": [0.15093873237651553, 0.14813448171146582, 0.1525722149291325]}, {\"tarea\": \"5dbc8537\", \"estrato\": \"barata\", \"celdas\": 535, \"n_tok\": 1897, \"exacta\": false, \"nll_celda\": 0.7158335462908878, \"nll_celda_med3\": 0.7082214925890771, \"nll_vistas\": [0.7158335462908878, 0.6977253851489486, 0.7082214925890771]}, {\"tarea\": \"291dc1e1\", \"estrato\": \"barata\", \"celdas\": 444, \"n_tok\": 1492, \"exacta\": false, \"nll_celda\": 0.7150793676977759, \"nll_celda_med3\": 0.7582906087239584, \"nll_vistas\": [0.7150793676977759, 0.7582906087239584, 0.7982812142586923]}, {\"tarea\": \"f931b4a8\", \"estrato\": \"barata\", \"celdas\": 528, \"n_tok\": 1255, \"exacta\": false, \"nll_celda\": 0.4125958066998106, \"nll_celda_med3\": 0.45670133648496686, \"nll_vistas\": [0.4125958066998106, 0.45670133648496686, 0.4608733148285837]}, {\"tarea\": \"f560132c\", \"estrato\": \"barata\", \"celdas\": 389, \"n_tok\": 2210, \"exacta\": false, \"nll_celda\": 0.637962400147112, \"nll_celda_med3\": 0.6561956724348289, \"nll_vistas\": [0.637962400147112, 0.6645928311776992, 0.6561956724348289]}, {\"tarea\": \"271d71e2\", \"estrato\": \"barata\", \"celdas\": 1173, \"n_tok\": 2496, \"exacta\": false, \"nll_celda\": 0.1809046908717631, \"nll_celda_med3\": 0.19273955336210571, \"nll_vistas\": [0.1809046908717631, 0.19273955336210571, 0.20673864028033087]}, {\"tarea\": \"db695cfb\", \"estrato\": \"media\", \"celdas\": 1222, \"n_tok\": 2642, \"exacta\": false, \"nll_celda\": 0.40344175847579533, \"nll_celda_med3\": 0.40344175847579533, \"nll_vistas\": [0.40344175847579533, 0.4485838245448036, 0.37502647187627863]}, {\"tarea\": \"7491f3cf\", \"estrato\": \"media\", \"celdas\": 875, \"n_tok\": 1850, \"exacta\": false, \"nll_celda\": 0.1408712419782366, \"nll_celda_med3\": 0.15259800502232143, \"nll_vistas\": [0.1408712419782366, 0.15259800502232143, 0.15879466029575892]}, {\"tarea\": \"abc82100\", \"estrato\": \"media\", \"celdas\": 1114, \"n_tok\": 2394, \"exacta\": false, \"nll_celda\": 0.27554376078251236, \"nll_celda_med3\": 0.2788620679982047, \"nll_vistas\": [0.27554376078251236, 0.2788620679982047, 0.29655566121346216]}, {\"tarea\": \"b6f77b65\", \"estrato\": \"media\", \"celdas\": 864, \"n_tok\": 1908, \"exacta\": false, \"nll_celda\": 0.27541928821139866, \"nll_celda_med3\": 0.20651273374204282, \"nll_vistas\": [0.27541928821139866, 0.20651273374204282, 0.20377367514151115]}, {\"tarea\": \"800d221b\", \"estrato\": \"media\", \"celdas\": 1656, \"n_tok\": 3488, \"exacta\": false, \"nll_celda\": 0.22955270665855224, \"nll_celda_med3\": 0.21300924918501848, \"nll_vistas\": [0.22955270665855224, 0.21188223649914137, 0.21300924918501848]}, {\"tarea\": \"8f3a5a89\", \"estrato\": \"media\", \"celdas\": 900, \"n_tok\": 1940, \"exacta\": false, \"nll_celda\": 0.24866658528645832, \"nll_celda_med3\": 0.24866658528645832, \"nll_vistas\": [0.24866658528645832, 0.2901529947916667, 0.21961829291449653]}, {\"tarea\": \"8698868d\", \"estrato\": \"media\", \"celdas\": 622, \"n_tok\": 2856, \"exacta\": false, \"nll_celda\": 0.4167295989499598, \"nll_celda_med3\": 0.41660026722015675, \"nll_vistas\": [0.4167295989499598, 0.4162155102303557, 0.41660026722015675]}, {\"tarea\": \"409aa875\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.18062976002693176, \"nll_celda_med3\": 0.1768522709608078, \"nll_vistas\": [0.18062976002693176, 0.1768522709608078, 0.16461728513240814]}, {\"tarea\": \"d59b0160\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.11737114936113358, \"nll_celda_med3\": 0.11737114936113358, \"nll_vistas\": [0.11737114936113358, 0.12481385469436646, 0.10960610210895538]}, {\"tarea\": \"e376de54\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.10243038833141327, \"nll_celda_med3\": 0.10899168252944946, \"nll_vistas\": [0.10243038833141327, 0.10899168252944946, 0.10900511592626572]}, {\"tarea\": \"58490d8a\", \"estrato\": \"media\", \"celdas\": 257, \"n_tok\": 2334, \"exacta\": false, \"nll_celda\": 0.46702863641286174, \"nll_celda_med3\": 0.5072103418728721, \"nll_vistas\": [0.46702863641286174, 0.5327028504605423, 0.5072103418728721]}, {\"tarea\": \"6ffbe589\", \"estrato\": \"media\", \"celdas\": 538, \"n_tok\": 2288, \"exacta\": false, \"nll_celda\": 0.5910221936534329, \"nll_celda_med3\": 0.5910221936534329, \"nll_vistas\": [0.5910221936534329, 0.5764314445864313, 0.6306035492056807]}], \"train\": {\"seq_hechas\": 5315, \"pasos\": 664, \"tok\": 9170128, \"lab\": 3589545, \"s\": 10801, \"tok_s\": 849, \"ckpt\": true, \"mem_GB\": 23.2, \"curva\": [{\"paso\": 1, \"seq\": 8, \"tok\": 14744, \"loss\": 0.539749164134264, \"lr\": 6.060606060606061e-06, \"s\": 20}, {\"paso\": 25, \"seq\": 200, \"tok\": 352492, \"loss\": 0.38258829240415554, \"lr\": 7.87878787878788e-05, \"s\": 417}, {\"paso\": 50, \"seq\": 400, \"tok\": 696295, \"loss\": 0.34040618060622363, \"lr\": 9.994021313919628e-05, \"s\": 823}, {\"paso\": 75, \"seq\": 600, \"tok\": 1038692, \"loss\": 0.27915394531562926, \"lr\": 9.96354437049027e-05, \"s\": 1225}, {\"paso\": 100, \"seq\": 800, \"tok\": 1381911, \"loss\": 0.30626685892231764, \"lr\": 9.907402550915433e-05, \"s\": 1630}, {\"paso\": 125, \"seq\": 1000, \"tok\": 1731559, \"loss\": 0.24880249405046925, \"lr\": 9.825886146313402e-05, \"s\": 2042}, {\"paso\": 150, \"seq\": 1200, \"tok\": 2074380, \"loss\": 0.2597964790440164, \"lr\": 9.719416651541839e-05, \"s\": 2446}, {\"paso\": 175, \"seq\": 1400, \"tok\": 2420562, \"loss\": 0.27595220401417464, \"lr\": 9.588544585784741e-05, \"s\": 2853}, {\"paso\": 200, \"seq\": 1600, \"tok\": 2765066, \"loss\": 0.26507700188085437, \"lr\": 9.433946645996514e-05, \"s\": 3261}, {\"paso\": 225, \"seq\": 1800, \"tok\": 3114966, \"loss\": 0.25006011261139066, \"lr\": 9.256422207921757e-05, \"s\": 3672}, {\"paso\": 250, \"seq\": 2000, \"tok\": 3455377, \"loss\": 0.30252157879061997, \"lr\": 9.056889192782866e-05, \"s\": 4074}, {\"paso\": 275, \"seq\": 2200, \"tok\": 3793824, \"loss\": 0.28189528309274464, \"lr\": 8.83637932100747e-05, \"s\": 4470}, {\"paso\": 300, \"seq\": 2400, \"tok\": 4139530, \"loss\": 0.25944113087840376, \"lr\": 8.596032777537123e-05, \"s\": 4878}, {\"paso\": 325, \"seq\": 2600, \"tok\": 4487131, \"loss\": 0.28898475609719754, \"lr\": 8.337092316301223e-05, \"s\": 5286}, {\"paso\": 350, \"seq\": 2800, \"tok\": 4834614, \"loss\": 0.27265949700493364, \"lr\": 8.060896834339993e-05, \"s\": 5696}, {\"paso\": 375, \"seq\": 3000, \"tok\": 5175747, \"loss\": 0.2677692821621895, \"lr\": 7.768874448802665e-05, \"s\": 6096}, {\"paso\": 400, \"seq\": 3200, \"tok\": 5514514, \"loss\": 0.26202524298336355, \"lr\": 7.462535112617452e-05, \"s\": 6496}, {\"paso\": 425, \"seq\": 3400, \"tok\": 5859385, \"loss\": 0.2761409277608618, \"lr\": 7.143462807015271e-05, \"s\": 6901}, {\"paso\": 450, \"seq\": 3600, \"tok\": 6206190, \"loss\": 0.2670974115468562, \"lr\": 6.81330735127716e-05, \"s\": 7310}, {\"paso\": 475, \"seq\": 3800, \"tok\": 6552652, \"loss\": 0.2547547717485577, \"lr\": 6.473775872054521e-05, \"s\": 7718}, {\"paso\": 500, \"seq\": 4000, \"tok\": 6894608, \"loss\": 0.24855377110652627, \"lr\": 6.12662397637155e-05, \"s\": 8121}, {\"paso\": 525, \"seq\": 4200, \"tok\": 7240934, \"loss\": 0.2803166220383719, \"lr\": 5.773646673951406e-05, \"s\": 8528}, {\"paso\": 550, \"seq\": 4400, \"tok\": 7586901, \"loss\": 0.24206583869177847, \"lr\": 5.4166690958038265e-05, \"s\": 8936}, {\"paso\": 575, \"seq\": 4600, \"tok\": 7929323, \"loss\": 0.25026059940457346, \"lr\": 5.057537057065338e-05, \"s\": 9338}, {\"paso\": 600, \"seq\": 4800, \"tok\": 8272931, \"loss\": 0.20647381878923624, \"lr\": 4.6981075128885693e-05, \"s\": 9743}, {\"paso\": 625, \"seq\": 5000, \"tok\": 8623146, \"loss\": 0.22052199121331795, \"lr\": 4.340238956730169e-05, \"s\": 10156}, {\"paso\": 650, \"seq\": 5200, \"tok\": 8970085, \"loss\": 0.23878908832091839, \"lr\": 3.985781810684631e-05, \"s\": 10566}]}, \"eval_post\": [{\"tarea\": \"e8686506\", \"estrato\": \"barata\", \"celdas\": 105, \"n_tok\": 748, \"exacta\": false, \"nll_celda\": 0.7635761079334077, \"nll_celda_med3\": 0.7637201218377976, \"nll_vistas\": [0.7635761079334077, 0.7637201218377976, 0.7882494245256696]}, {\"tarea\": \"20270e3b\", \"estrato\": \"barata\", \"celdas\": 226, \"n_tok\": 709, \"exacta\": false, \"nll_celda\": 0.40411086631032217, \"nll_celda_med3\": 0.4097065883400166, \"nll_vistas\": [0.40411086631032217, 0.4097065883400166, 0.4165371244987555]}, {\"tarea\": \"7b5033c1\", \"estrato\": \"barata\", \"celdas\": 71, \"n_tok\": 976, \"exacta\": false, \"nll_celda\": 0.579116230279627, \"nll_celda_med3\": 0.6690448707258198, \"nll_vistas\": [0.579116230279627, 0.830459540998432, 0.6690448707258198]}, {\"tarea\": \"b0039139\", \"estrato\": \"barata\", \"celdas\": 212, \"n_tok\": 957, \"exacta\": false, \"nll_celda\": 0.47019433075526973, \"nll_celda_med3\": 0.485978864273935, \"nll_vistas\": [0.47019433075526973, 0.485978864273935, 0.49342918395996094]}, {\"tarea\": \"78332cb0\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 930, \"exacta\": false, \"nll_celda\": 0.1867723846435547, \"nll_celda_med3\": 0.20718685150146485, \"nll_vistas\": [0.1867723846435547, 0.20718685150146485, 0.24581119537353516]}, {\"tarea\": \"28a6681f\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.12465126037597657, \"nll_celda_med3\": 0.15459521293640135, \"nll_vistas\": [0.12465126037597657, 0.15459521293640135, 0.15952957153320313]}, {\"tarea\": \"d35bdbdc\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.15830423355102538, \"nll_celda_med3\": 0.16251678466796876, \"nll_vistas\": [0.15830423355102538, 0.17141548156738282, 0.16251678466796876]}, {\"tarea\": \"dbff022c\", \"estrato\": \"barata\", \"celdas\": 473, \"n_tok\": 1048, \"exacta\": false, \"nll_celda\": 0.12629874715341322, \"nll_celda_med3\": 0.10416263959372522, \"nll_vistas\": [0.12629874715341322, 0.10416263959372522, 0.0983642191009844]}, {\"tarea\": \"4c3d4a41\", \"estrato\": \"barata\", \"celdas\": 480, \"n_tok\": 1026, \"exacta\": false, \"nll_celda\": 0.13666610717773436, \"nll_celda_med3\": 0.11338399251302084, \"nll_vistas\": [0.13666610717773436, 0.11338399251302084, 0.0999370257059733]}, {\"tarea\": \"446ef5d2\", \"estrato\": \"barata\", \"celdas\": 1016, \"n_tok\": 2146, \"exacta\": false, \"nll_celda\": 0.09615422046090674, \"nll_celda_med3\": 0.11444989902766671, \"nll_vistas\": [0.09615422046090674, 0.11444989902766671, 0.12092788576141117]}, {\"tarea\": \"dd6b8c4b\", \"estrato\": \"barata\", \"celdas\": 484, \"n_tok\": 1080, \"exacta\": false, \"nll_celda\": 0.14301426154522856, \"nll_celda_med3\": 0.1473094530342039, \"nll_vistas\": [0.14301426154522856, 0.1473094530342039, 0.147362228267449]}, {\"tarea\": \"3dc255db\", \"estrato\": \"barata\", \"celdas\": 544, \"n_tok\": 1204, \"exacta\": false, \"nll_celda\": 0.20940073798684514, \"nll_celda_med3\": 0.20130816627951229, \"nll_vistas\": [0.20940073798684514, 0.19221120722153606, 0.20130816627951229]}, {\"tarea\": \"16de56c4\", \"estrato\": \"barata\", \"celdas\": 582, \"n_tok\": 1262, \"exacta\": false, \"nll_celda\": 0.22365638234770993, \"nll_celda_med3\": 0.25028965481368126, \"nll_vistas\": [0.22365638234770993, 0.2852688032327239, 0.25028965481368126]}, {\"tarea\": \"97d7923e\", \"estrato\": \"barata\", \"celdas\": 600, \"n_tok\": 1304, \"exacta\": false, \"nll_celda\": 0.0754028574625651, \"nll_celda_med3\": 0.053101393381754555, \"nll_vistas\": [0.0754028574625651, 0.053101393381754555, 0.05181301116943359]}, {\"tarea\": \"2b83f449\", \"estrato\": \"barata\", \"celdas\": 746, \"n_tok\": 1596, \"exacta\": false, \"nll_celda\": 0.28433121177530163, \"nll_celda_med3\": 0.2573396002639075, \"nll_vistas\": [0.28433121177530163, 0.23143455362192109, 0.2573396002639075]}, {\"tarea\": \"bf45cf4b\", \"estrato\": \"barata\", \"celdas\": 985, \"n_tok\": 1791, \"exacta\": false, \"nll_celda\": 0.10460693049551872, \"nll_celda_med3\": 0.09674684166303141, \"nll_vistas\": [0.10460693049551872, 0.09292947991850413, 0.09674684166303141]}, {\"tarea\": \"faa9f03d\", \"estrato\": \"barata\", \"celdas\": 832, \"n_tok\": 1800, \"exacta\": false, \"nll_celda\": 0.14672425160041222, \"nll_celda_med3\": 0.15700604365422174, \"nll_vistas\": [0.14672425160041222, 0.157060641508836, 0.15700604365422174]}, {\"tarea\": \"eee78d87\", \"estrato\": \"barata\", \"celdas\": 1024, \"n_tok\": 1280, \"exacta\": false, \"nll_celda\": 0.12524759769439697, \"nll_celda_med3\": 0.12853583693504333, \"nll_vistas\": [0.12524759769439697, 0.12853583693504333, 0.1331929713487625]}, {\"tarea\": \"31f7f899\", \"estrato\": \"barata\", \"celdas\": 732, \"n_tok\": 1592, \"exacta\": false, \"nll_celda\": 0.12069208113873592, \"nll_celda_med3\": 0.10789850370480063, \"nll_vistas\": [0.12069208113873592, 0.10789850370480063, 0.10488022350874103]}, {\"tarea\": \"5dbc8537\", \"estrato\": \"barata\", \"celdas\": 535, \"n_tok\": 1897, \"exacta\": false, \"nll_celda\": 0.4361301029954001, \"nll_celda_med3\": 0.4112601021739924, \"nll_vistas\": [0.4361301029954001, 0.4112601021739924, 0.39479107723057827]}, {\"tarea\": \"291dc1e1\", \"estrato\": \"barata\", \"celdas\": 444, \"n_tok\": 1492, \"exacta\": false, \"nll_celda\": 0.5046100100955447, \"nll_celda_med3\": 0.6158573734867681, \"nll_vistas\": [0.5046100100955447, 0.632552550719665, 0.6158573734867681]}, {\"tarea\": \"f931b4a8\", \"estrato\": \"barata\", \"celdas\": 528, \"n_tok\": 1255, \"exacta\": false, \"nll_celda\": 0.23261971907182175, \"nll_celda_med3\": 0.30970368240818835, \"nll_vistas\": [0.23261971907182175, 0.30970368240818835, 0.3399536826393821]}, {\"tarea\": \"f560132c\", \"estrato\": \"barata\", \"celdas\": 389, \"n_tok\": 2210, \"exacta\": false, \"nll_celda\": 0.50238437211299, \"nll_celda_med3\": 0.4980835902353792, \"nll_vistas\": [0.50238437211299, 0.46764030554607167, 0.4980835902353792]}, {\"tarea\": \"271d71e2\", \"estrato\": \"barata\", \"celdas\": 1173, \"n_tok\": 2496, \"exacta\": false, \"nll_celda\": 0.1329904151389666, \"nll_celda_med3\": 0.13554094839787137, \"nll_vistas\": [0.1329904151389666, 0.13554094839787137, 0.15949886026081628]}, {\"tarea\": \"db695cfb\", \"estrato\": \"media\", \"celdas\": 1222, \"n_tok\": 2642, \"exacta\": false, \"nll_celda\": 0.21687204513924407, \"nll_celda_med3\": 0.2023870340150624, \"nll_vistas\": [0.21687204513924407, 0.1927400829358889, 0.2023870340150624]}, {\"tarea\": \"7491f3cf\", \"estrato\": \"media\", \"celdas\": 875, \"n_tok\": 1850, \"exacta\": false, \"nll_celda\": 0.09289300537109375, \"nll_celda_med3\": 0.09416325160435268, \"nll_vistas\": [0.09289300537109375, 0.10040373011997768, 0.09416325160435268]}, {\"tarea\": \"abc82100\", \"estrato\": \"media\", \"celdas\": 1114, \"n_tok\": 2394, \"exacta\": false, \"nll_celda\": 0.22764854499638829, \"nll_celda_med3\": 0.2384821071658999, \"nll_vistas\": [0.22764854499638829, 0.24188144759173025, 0.2384821071658999]}, {\"tarea\": \"b6f77b65\", \"estrato\": \"media\", \"celdas\": 864, \"n_tok\": 1908, \"exacta\": false, \"nll_celda\": 0.23004595438639322, \"nll_celda_med3\": 0.1685713662041558, \"nll_vistas\": [0.23004595438639322, 0.1685713662041558, 0.1420330118249964]}, {\"tarea\": \"800d221b\", \"estrato\": \"media\", \"celdas\": 1656, \"n_tok\": 3488, \"exacta\": false, \"nll_celda\": 0.13644326251486075, \"nll_celda_med3\": 0.13641034923313897, \"nll_vistas\": [0.13644326251486075, 0.11333476633265399, 0.13641034923313897]}, {\"tarea\": \"8f3a5a89\", \"estrato\": \"media\", \"celdas\": 900, \"n_tok\": 1940, \"exacta\": false, \"nll_celda\": 0.16565907796223958, \"nll_celda_med3\": 0.16390231662326388, \"nll_vistas\": [0.16565907796223958, 0.16390231662326388, 0.16088626437717013]}, {\"tarea\": \"8698868d\", \"estrato\": \"media\", \"celdas\": 622, \"n_tok\": 2856, \"exacta\": false, \"nll_celda\": 0.23522299125646856, \"nll_celda_med3\": 0.24382257001576316, \"nll_vistas\": [0.23522299125646856, 0.2583303666191469, 0.24382257001576316]}, {\"tarea\": \"409aa875\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.13199308514595032, \"nll_celda_med3\": 0.13691027462482452, \"nll_vistas\": [0.13199308514595032, 0.13691027462482452, 0.14334052801132202]}, {\"tarea\": \"d59b0160\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.0676930844783783, \"nll_celda_med3\": 0.061825476586818695, \"nll_vistas\": [0.0676930844783783, 0.054659370332956314, 0.061825476586818695]}, {\"tarea\": \"e376de54\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.07953470945358276, \"nll_celda_med3\": 0.07953470945358276, \"nll_vistas\": [0.07953470945358276, 0.08291573822498322, 0.07573969662189484]}, {\"tarea\": \"58490d8a\", \"estrato\": \"media\", \"celdas\": 257, \"n_tok\": 2334, \"exacta\": false, \"nll_celda\": 0.18951901387611716, \"nll_celda_med3\": 0.26657116366731515, \"nll_vistas\": [0.18951901387611716, 0.2959071652898529, 0.26657116366731515]}, {\"tarea\": \"6ffbe589\", \"estrato\": \"media\", \"celdas\": 538, \"n_tok\": 2288, \"exacta\": false, \"nll_celda\": 0.47274522178678263, \"nll_celda_med3\": 0.47274522178678263, \"nll_vistas\": [0.47274522178678263, 0.46357131447490707, 0.4832294563378543]}]}, \"q3_s2\": {\"arm\": \"q3\", \"seed\": 2, \"modelo\": \"/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1\", \"clase\": \"AutoModelForCausalLM\", \"ids16\": [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 198, 872, 77091, 151643, 151644, 151645], \"tied\": true, \"H\": 2560, \"n_seq\": 9000, \"n_tok\": 15493036, \"n_lab\": 6057102, \"params_tr_M\": 132.2, \"lineales\": [\"down_proj\", \"gate_proj\", \"k_proj\", \"o_proj\", \"q_proj\", \"up_proj\", \"v_proj\"], \"cfg\": {\"lr\": 0.0001, \"accum\": 8, \"r\": 64, \"alpha\": 32, \"max_len\": 2048, \"train_seconds\": 10800.0}, \"fla\": true, \"transformers\": \"5.16.1\", \"eval_pre\": [{\"tarea\": \"e8686506\", \"estrato\": \"barata\", \"celdas\": 105, \"n_tok\": 748, \"exacta\": false, \"nll_celda\": 1.4696684337797619, \"nll_celda_med3\": 1.3998081752232143, \"nll_vistas\": [1.4696684337797619, 1.3540909540085566, 1.3998081752232143]}, {\"tarea\": \"20270e3b\", \"estrato\": \"barata\", \"celdas\": 226, \"n_tok\": 709, \"exacta\": false, \"nll_celda\": 0.6027137283730296, \"nll_celda_med3\": 0.6064603687387652, \"nll_vistas\": [0.6027137283730296, 0.6064603687387652, 0.6350239441458103]}, {\"tarea\": \"7b5033c1\", \"estrato\": \"barata\", \"celdas\": 71, \"n_tok\": 976, \"exacta\": false, \"nll_celda\": 1.5271193813270247, \"nll_celda_med3\": 1.6248020655672315, \"nll_vistas\": [1.5271193813270247, 1.672822119484485, 1.6248020655672315]}, {\"tarea\": \"b0039139\", \"estrato\": \"barata\", \"celdas\": 212, \"n_tok\": 957, \"exacta\": false, \"nll_celda\": 0.7913860824872863, \"nll_celda_med3\": 0.7913860824872863, \"nll_vistas\": [0.7913860824872863, 0.7839133784456073, 0.8038517214217276]}, {\"tarea\": \"78332cb0\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 930, \"exacta\": false, \"nll_celda\": 0.3866337585449219, \"nll_celda_med3\": 0.4537655258178711, \"nll_vistas\": [0.3866337585449219, 0.4537655258178711, 0.5057917785644531]}, {\"tarea\": \"28a6681f\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.2178778839111328, \"nll_celda_med3\": 0.22701530456542968, \"nll_vistas\": [0.2178778839111328, 0.2667096710205078, 0.22701530456542968]}, {\"tarea\": \"d35bdbdc\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.3074209213256836, \"nll_celda_med3\": 0.3194326972961426, \"nll_vistas\": [0.3074209213256836, 0.3194326972961426, 0.35251895904541014]}, {\"tarea\": \"dbff022c\", \"estrato\": \"barata\", \"celdas\": 473, \"n_tok\": 1048, \"exacta\": false, \"nll_celda\": 0.18631502437793176, \"nll_celda_med3\": 0.20239751384323798, \"nll_vistas\": [0.18631502437793176, 0.20239751384323798, 0.21213220189036072]}, {\"tarea\": \"4c3d4a41\", \"estrato\": \"barata\", \"celdas\": 480, \"n_tok\": 1026, \"exacta\": false, \"nll_celda\": 0.17859843571980794, \"nll_celda_med3\": 0.18151445388793946, \"nll_vistas\": [0.17859843571980794, 0.18151445388793946, 0.19257771174112956]}, {\"tarea\": \"446ef5d2\", \"estrato\": \"barata\", \"celdas\": 1016, \"n_tok\": 2146, \"exacta\": false, \"nll_celda\": 0.15325765534648744, \"nll_celda_med3\": 0.1737351304902805, \"nll_vistas\": [0.15325765534648744, 0.18151092529296875, 0.1737351304902805]}, {\"tarea\": \"dd6b8c4b\", \"estrato\": \"barata\", \"celdas\": 484, \"n_tok\": 1080, \"exacta\": false, \"nll_celda\": 0.19546146235190148, \"nll_celda_med3\": 0.19546146235190148, \"nll_vistas\": [0.19546146235190148, 0.19415380935038418, 0.22238597396976692]}, {\"tarea\": \"3dc255db\", \"estrato\": \"barata\", \"celdas\": 544, \"n_tok\": 1204, \"exacta\": false, \"nll_celda\": 0.27532117507036996, \"nll_celda_med3\": 0.27955470365636487, \"nll_vistas\": [0.27532117507036996, 0.27955470365636487, 0.29760585111730237]}, {\"tarea\": \"16de56c4\", \"estrato\": \"barata\", \"celdas\": 582, \"n_tok\": 1262, \"exacta\": false, \"nll_celda\": 0.34055152709541453, \"nll_celda_med3\": 0.3683308080299613, \"nll_vistas\": [0.34055152709541453, 0.36843384418290914, 0.3683308080299613]}, {\"tarea\": \"97d7923e\", \"estrato\": \"barata\", \"celdas\": 600, \"n_tok\": 1304, \"exacta\": false, \"nll_celda\": 0.09632901509602865, \"nll_celda_med3\": 0.09985081354777019, \"nll_vistas\": [0.09632901509602865, 0.09985081354777019, 0.12942376454671223]}, {\"tarea\": \"2b83f449\", \"estrato\": \"barata\", \"celdas\": 746, \"n_tok\": 1596, \"exacta\": false, \"nll_celda\": 0.4448564135676734, \"nll_celda_med3\": 0.4448564135676734, \"nll_vistas\": [0.4448564135676734, 0.4003213263708529, 0.4914749186416094]}, {\"tarea\": \"bf45cf4b\", \"estrato\": \"barata\", \"celdas\": 985, \"n_tok\": 1791, \"exacta\": false, \"nll_celda\": 0.22940069673025063, \"nll_celda_med3\": 0.21452362525281568, \"nll_vistas\": [0.22940069673025063, 0.21452362525281568, 0.20762519642786326]}, {\"tarea\": \"faa9f03d\", \"estrato\": \"barata\", \"celdas\": 832, \"n_tok\": 1800, \"exacta\": false, \"nll_celda\": 0.19694962868323693, \"nll_celda_med3\": 0.20706682938795823, \"nll_vistas\": [0.19694962868323693, 0.20706682938795823, 0.23652780972994292]}, {\"tarea\": \"eee78d87\", \"estrato\": \"barata\", \"celdas\": 1024, \"n_tok\": 1280, \"exacta\": false, \"nll_celda\": 0.13134634494781494, \"nll_celda_med3\": 0.14614208042621613, \"nll_vistas\": [0.13134634494781494, 0.14614208042621613, 0.15498989820480347]}, {\"tarea\": \"31f7f899\", \"estrato\": \"barata\", \"celdas\": 732, \"n_tok\": 1592, \"exacta\": false, \"nll_celda\": 0.15093873237651553, \"nll_celda_med3\": 0.15093873237651553, \"nll_vistas\": [0.15093873237651553, 0.14813448171146582, 0.1525722149291325]}, {\"tarea\": \"5dbc8537\", \"estrato\": \"barata\", \"celdas\": 535, \"n_tok\": 1897, \"exacta\": false, \"nll_celda\": 0.7158335462908878, \"nll_celda_med3\": 0.7082214925890771, \"nll_vistas\": [0.7158335462908878, 0.6977253851489486, 0.7082214925890771]}, {\"tarea\": \"291dc1e1\", \"estrato\": \"barata\", \"celdas\": 444, \"n_tok\": 1492, \"exacta\": false, \"nll_celda\": 0.7150793676977759, \"nll_celda_med3\": 0.7582906087239584, \"nll_vistas\": [0.7150793676977759, 0.7582906087239584, 0.7982812142586923]}, {\"tarea\": \"f931b4a8\", \"estrato\": \"barata\", \"celdas\": 528, \"n_tok\": 1255, \"exacta\": false, \"nll_celda\": 0.4125958066998106, \"nll_celda_med3\": 0.45670133648496686, \"nll_vistas\": [0.4125958066998106, 0.45670133648496686, 0.4608733148285837]}, {\"tarea\": \"f560132c\", \"estrato\": \"barata\", \"celdas\": 389, \"n_tok\": 2210, \"exacta\": false, \"nll_celda\": 0.637962400147112, \"nll_celda_med3\": 0.6561956724348289, \"nll_vistas\": [0.637962400147112, 0.6645928311776992, 0.6561956724348289]}, {\"tarea\": \"271d71e2\", \"estrato\": \"barata\", \"celdas\": 1173, \"n_tok\": 2496, \"exacta\": false, \"nll_celda\": 0.1809046908717631, \"nll_celda_med3\": 0.19273955336210571, \"nll_vistas\": [0.1809046908717631, 0.19273955336210571, 0.20673864028033087]}, {\"tarea\": \"db695cfb\", \"estrato\": \"media\", \"celdas\": 1222, \"n_tok\": 2642, \"exacta\": false, \"nll_celda\": 0.40344175847579533, \"nll_celda_med3\": 0.40344175847579533, \"nll_vistas\": [0.40344175847579533, 0.4485838245448036, 0.37502647187627863]}, {\"tarea\": \"7491f3cf\", \"estrato\": \"media\", \"celdas\": 875, \"n_tok\": 1850, \"exacta\": false, \"nll_celda\": 0.1408712419782366, \"nll_celda_med3\": 0.15259800502232143, \"nll_vistas\": [0.1408712419782366, 0.15259800502232143, 0.15879466029575892]}, {\"tarea\": \"abc82100\", \"estrato\": \"media\", \"celdas\": 1114, \"n_tok\": 2394, \"exacta\": false, \"nll_celda\": 0.27554376078251236, \"nll_celda_med3\": 0.2788620679982047, \"nll_vistas\": [0.27554376078251236, 0.2788620679982047, 0.29655566121346216]}, {\"tarea\": \"b6f77b65\", \"estrato\": \"media\", \"celdas\": 864, \"n_tok\": 1908, \"exacta\": false, \"nll_celda\": 0.27541928821139866, \"nll_celda_med3\": 0.20651273374204282, \"nll_vistas\": [0.27541928821139866, 0.20651273374204282, 0.20377367514151115]}, {\"tarea\": \"800d221b\", \"estrato\": \"media\", \"celdas\": 1656, \"n_tok\": 3488, \"exacta\": false, \"nll_celda\": 0.22955270665855224, \"nll_celda_med3\": 0.21300924918501848, \"nll_vistas\": [0.22955270665855224, 0.21188223649914137, 0.21300924918501848]}, {\"tarea\": \"8f3a5a89\", \"estrato\": \"media\", \"celdas\": 900, \"n_tok\": 1940, \"exacta\": false, \"nll_celda\": 0.24866658528645832, \"nll_celda_med3\": 0.24866658528645832, \"nll_vistas\": [0.24866658528645832, 0.2901529947916667, 0.21961829291449653]}, {\"tarea\": \"8698868d\", \"estrato\": \"media\", \"celdas\": 622, \"n_tok\": 2856, \"exacta\": false, \"nll_celda\": 0.4167295989499598, \"nll_celda_med3\": 0.41660026722015675, \"nll_vistas\": [0.4167295989499598, 0.4162155102303557, 0.41660026722015675]}, {\"tarea\": \"409aa875\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.18062976002693176, \"nll_celda_med3\": 0.1768522709608078, \"nll_vistas\": [0.18062976002693176, 0.1768522709608078, 0.16461728513240814]}, {\"tarea\": \"d59b0160\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.11737114936113358, \"nll_celda_med3\": 0.11737114936113358, \"nll_vistas\": [0.11737114936113358, 0.12481385469436646, 0.10960610210895538]}, {\"tarea\": \"e376de54\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.10243038833141327, \"nll_celda_med3\": 0.10899168252944946, \"nll_vistas\": [0.10243038833141327, 0.10899168252944946, 0.10900511592626572]}, {\"tarea\": \"58490d8a\", \"estrato\": \"media\", \"celdas\": 257, \"n_tok\": 2334, \"exacta\": false, \"nll_celda\": 0.46702863641286174, \"nll_celda_med3\": 0.5072103418728721, \"nll_vistas\": [0.46702863641286174, 0.5327028504605423, 0.5072103418728721]}, {\"tarea\": \"6ffbe589\", \"estrato\": \"media\", \"celdas\": 538, \"n_tok\": 2288, \"exacta\": false, \"nll_celda\": 0.5910221936534329, \"nll_celda_med3\": 0.5910221936534329, \"nll_vistas\": [0.5910221936534329, 0.5764314445864313, 0.6306035492056807]}], \"train\": {\"seq_hechas\": 5345, \"pasos\": 668, \"tok\": 9199142, \"lab\": 3594844, \"s\": 10802, \"tok_s\": 852, \"ckpt\": true, \"mem_GB\": 23.1, \"curva\": [{\"paso\": 1, \"seq\": 8, \"tok\": 14693, \"loss\": 0.5755404755473137, \"lr\": 6.060606060606061e-06, \"s\": 18}, {\"paso\": 25, \"seq\": 200, \"tok\": 347057, \"loss\": 0.3763543478756522, \"lr\": 7.87878787878788e-05, \"s\": 407}, {\"paso\": 50, \"seq\": 400, \"tok\": 692590, \"loss\": 0.3107775198109448, \"lr\": 9.994021313919628e-05, \"s\": 812}, {\"paso\": 75, \"seq\": 600, \"tok\": 1037498, \"loss\": 0.30098784691188485, \"lr\": 9.96354437049027e-05, \"s\": 1216}, {\"paso\": 100, \"seq\": 800, \"tok\": 1387361, \"loss\": 0.30507887529209254, \"lr\": 9.907402550915433e-05, \"s\": 1629}, {\"paso\": 125, \"seq\": 1000, \"tok\": 1726959, \"loss\": 0.260679567521438, \"lr\": 9.825886146313402e-05, \"s\": 2026}, {\"paso\": 150, \"seq\": 1200, \"tok\": 2066848, \"loss\": 0.2535198533162475, \"lr\": 9.719416651541839e-05, \"s\": 2425}, {\"paso\": 175, \"seq\": 1400, \"tok\": 2413116, \"loss\": 0.26589790703728794, \"lr\": 9.588544585784741e-05, \"s\": 2831}, {\"paso\": 200, \"seq\": 1600, \"tok\": 2754931, \"loss\": 0.27168641563504936, \"lr\": 9.433946645996514e-05, \"s\": 3233}, {\"paso\": 225, \"seq\": 1800, \"tok\": 3098780, \"loss\": 0.23457373588811606, \"lr\": 9.256422207921757e-05, \"s\": 3636}, {\"paso\": 250, \"seq\": 2000, \"tok\": 3442422, \"loss\": 0.3070035248668864, \"lr\": 9.056889192782866e-05, \"s\": 4040}, {\"paso\": 275, \"seq\": 2200, \"tok\": 3787355, \"loss\": 0.2759091583732516, \"lr\": 8.83637932100747e-05, \"s\": 4445}, {\"paso\": 300, \"seq\": 2400, \"tok\": 4126638, \"loss\": 0.27472571686841546, \"lr\": 8.596032777537123e-05, \"s\": 4843}, {\"paso\": 325, \"seq\": 2600, \"tok\": 4471035, \"loss\": 0.27165080439299344, \"lr\": 8.337092316301223e-05, \"s\": 5247}, {\"paso\": 350, \"seq\": 2800, \"tok\": 4806807, \"loss\": 0.2900051845982671, \"lr\": 8.060896834339993e-05, \"s\": 5642}, {\"paso\": 375, \"seq\": 3000, \"tok\": 5154929, \"loss\": 0.281887326897122, \"lr\": 7.768874448802665e-05, \"s\": 6050}, {\"paso\": 400, \"seq\": 3200, \"tok\": 5498219, \"loss\": 0.2768496490595862, \"lr\": 7.462535112617452e-05, \"s\": 6454}, {\"paso\": 425, \"seq\": 3400, \"tok\": 5842434, \"loss\": 0.27780657827388494, \"lr\": 7.143462807015271e-05, \"s\": 6858}, {\"paso\": 450, \"seq\": 3600, \"tok\": 6184198, \"loss\": 0.27704320916906, \"lr\": 6.81330735127716e-05, \"s\": 7260}, {\"paso\": 475, \"seq\": 3800, \"tok\": 6526554, \"loss\": 0.26444451621267945, \"lr\": 6.473775872054521e-05, \"s\": 7661}, {\"paso\": 500, \"seq\": 4000, \"tok\": 6873344, \"loss\": 0.2665634387778118, \"lr\": 6.12662397637155e-05, \"s\": 8069}, {\"paso\": 525, \"seq\": 4200, \"tok\": 7221371, \"loss\": 0.2709433762496337, \"lr\": 5.773646673951406e-05, \"s\": 8478}, {\"paso\": 550, \"seq\": 4400, \"tok\": 7568919, \"loss\": 0.22882376830093562, \"lr\": 5.4166690958038265e-05, \"s\": 8888}, {\"paso\": 575, \"seq\": 4600, \"tok\": 7910168, \"loss\": 0.2485585919301957, \"lr\": 5.057537057065338e-05, \"s\": 9287}, {\"paso\": 600, \"seq\": 4800, \"tok\": 8261416, \"loss\": 0.21629052139353008, \"lr\": 4.6981075128885693e-05, \"s\": 9701}, {\"paso\": 625, \"seq\": 5000, \"tok\": 8610589, \"loss\": 0.22974257576745002, \"lr\": 4.340238956730169e-05, \"s\": 10111}, {\"paso\": 650, \"seq\": 5200, \"tok\": 8952635, \"loss\": 0.24232980047352612, \"lr\": 3.985781810684631e-05, \"s\": 10513}]}, \"eval_post\": [{\"tarea\": \"e8686506\", \"estrato\": \"barata\", \"celdas\": 105, \"n_tok\": 748, \"exacta\": false, \"nll_celda\": 0.6895846412295387, \"nll_celda_med3\": 0.7309618268694197, \"nll_vistas\": [0.6895846412295387, 0.7309618268694197, 0.7561321440197173]}, {\"tarea\": \"20270e3b\", \"estrato\": \"barata\", \"celdas\": 226, \"n_tok\": 709, \"exacta\": false, \"nll_celda\": 0.4161636217505531, \"nll_celda_med3\": 0.44560488135413784, \"nll_vistas\": [0.4161636217505531, 0.45224782217920356, 0.44560488135413784]}, {\"tarea\": \"7b5033c1\", \"estrato\": \"barata\", \"celdas\": 71, \"n_tok\": 976, \"exacta\": false, \"nll_celda\": 0.6254110201983385, \"nll_celda_med3\": 0.6843265479719135, \"nll_vistas\": [0.6254110201983385, 0.8289482761436785, 0.6843265479719135]}, {\"tarea\": \"b0039139\", \"estrato\": \"barata\", \"celdas\": 212, \"n_tok\": 957, \"exacta\": false, \"nll_celda\": 0.47122296747171655, \"nll_celda_med3\": 0.48665021500497496, \"nll_vistas\": [0.47122296747171655, 0.48665021500497496, 0.4992866516113281]}, {\"tarea\": \"78332cb0\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 930, \"exacta\": false, \"nll_celda\": 0.20702152252197265, \"nll_celda_med3\": 0.21676788330078126, \"nll_vistas\": [0.20702152252197265, 0.21676788330078126, 0.23856761932373047]}, {\"tarea\": \"28a6681f\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.1258897876739502, \"nll_celda_med3\": 0.1378653335571289, \"nll_vistas\": [0.1258897876739502, 0.15857085227966308, 0.1378653335571289]}, {\"tarea\": \"d35bdbdc\", \"estrato\": \"barata\", \"celdas\": 400, \"n_tok\": 904, \"exacta\": false, \"nll_celda\": 0.16761463165283202, \"nll_celda_med3\": 0.16761463165283202, \"nll_vistas\": [0.16761463165283202, 0.16153121948242188, 0.16903388977050782]}, {\"tarea\": \"dbff022c\", \"estrato\": \"barata\", \"celdas\": 473, \"n_tok\": 1048, \"exacta\": false, \"nll_celda\": 0.1249123184171842, \"nll_celda_med3\": 0.10746247370197708, \"nll_vistas\": [0.1249123184171842, 0.10746247370197708, 0.10713266771893169]}, {\"tarea\": \"4c3d4a41\", \"estrato\": \"barata\", \"celdas\": 480, \"n_tok\": 1026, \"exacta\": false, \"nll_celda\": 0.1272177537282308, \"nll_celda_med3\": 0.10939168930053711, \"nll_vistas\": [0.1272177537282308, 0.10939168930053711, 0.09988516171773275]}, {\"tarea\": \"446ef5d2\", \"estrato\": \"barata\", \"celdas\": 1016, \"n_tok\": 2146, \"exacta\": false, \"nll_celda\": 0.08763525805135411, \"nll_celda_med3\": 0.1159919453418161, \"nll_vistas\": [0.08763525805135411, 0.1159919453418161, 0.12188708688330463]}, {\"tarea\": \"dd6b8c4b\", \"estrato\": \"barata\", \"celdas\": 484, \"n_tok\": 1080, \"exacta\": false, \"nll_celda\": 0.16175344167662062, \"nll_celda_med3\": 0.15671739499431012, \"nll_vistas\": [0.16175344167662062, 0.15671739499431012, 0.15411633893478016]}, {\"tarea\": \"3dc255db\", \"estrato\": \"barata\", \"celdas\": 544, \"n_tok\": 1204, \"exacta\": false, \"nll_celda\": 0.20132878247429342, \"nll_celda_med3\": 0.20250570072847254, \"nll_vistas\": [0.20132878247429342, 0.20250570072847254, 0.20594159294577205]}, {\"tarea\": \"16de56c4\", \"estrato\": \"barata\", \"celdas\": 582, \"n_tok\": 1262, \"exacta\": false, \"nll_celda\": 0.22065438601569212, \"nll_celda_med3\": 0.26907878233395083, \"nll_vistas\": [0.22065438601569212, 0.2994974405085508, 0.26907878233395083]}, {\"tarea\": \"97d7923e\", \"estrato\": \"barata\", \"celdas\": 600, \"n_tok\": 1304, \"exacta\": false, \"nll_celda\": 0.06156228383382161, \"nll_celda_med3\": 0.06156228383382161, \"nll_vistas\": [0.06156228383382161, 0.06592578887939453, 0.04654501597086588]}, {\"tarea\": \"2b83f449\", \"estrato\": \"barata\", \"celdas\": 746, \"n_tok\": 1596, \"exacta\": false, \"nll_celda\": 0.2967629522165089, \"nll_celda_med3\": 0.25784066535192907, \"nll_vistas\": [0.2967629522165089, 0.23192058491642928, 0.25784066535192907]}, {\"tarea\": \"bf45cf4b\", \"estrato\": \"barata\", \"celdas\": 985, \"n_tok\": 1791, \"exacta\": false, \"nll_celda\": 0.11475569052139514, \"nll_celda_med3\": 0.10092130554509042, \"nll_vistas\": [0.11475569052139514, 0.10092130554509042, 0.1006993947295368]}, {\"tarea\": \"faa9f03d\", \"estrato\": \"barata\", \"celdas\": 832, \"n_tok\": 1800, \"exacta\": false, \"nll_celda\": 0.1533078413743239, \"nll_celda_med3\": 0.1533078413743239, \"nll_vistas\": [0.1533078413743239, 0.16060735629155085, 0.14827720935528094]}, {\"tarea\": \"eee78d87\", \"estrato\": \"barata\", \"celdas\": 1024, \"n_tok\": 1280, \"exacta\": false, \"nll_celda\": 0.12091486155986786, \"nll_celda_med3\": 0.13196581602096558, \"nll_vistas\": [0.12091486155986786, 0.13196581602096558, 0.13382205367088318]}, {\"tarea\": \"31f7f899\", \"estrato\": \"barata\", \"celdas\": 732, \"n_tok\": 1592, \"exacta\": false, \"nll_celda\": 0.09720857547280568, \"nll_celda_med3\": 0.10470975031618213, \"nll_vistas\": [0.09720857547280568, 0.10470975031618213, 0.11028353373209636]}, {\"tarea\": \"5dbc8537\", \"estrato\": \"barata\", \"celdas\": 535, \"n_tok\": 1897, \"exacta\": false, \"nll_celda\": 0.45615747754818925, \"nll_celda_med3\": 0.40210397310346085, \"nll_vistas\": [0.45615747754818925, 0.40210397310346085, 0.39848359009929907]}, {\"tarea\": \"291dc1e1\", \"estrato\": \"barata\", \"celdas\": 444, \"n_tok\": 1492, \"exacta\": false, \"nll_celda\": 0.4986027898015203, \"nll_celda_med3\": 0.5915034526103252, \"nll_vistas\": [0.4986027898015203, 0.6118819090697143, 0.5915034526103252]}, {\"tarea\": \"f931b4a8\", \"estrato\": \"barata\", \"celdas\": 528, \"n_tok\": 1255, \"exacta\": false, \"nll_celda\": 0.18637660055449515, \"nll_celda_med3\": 0.29127490881717566, \"nll_vistas\": [0.18637660055449515, 0.29127490881717566, 0.3249392364964341]}, {\"tarea\": \"f560132c\", \"estrato\": \"barata\", \"celdas\": 389, \"n_tok\": 2210, \"exacta\": false, \"nll_celda\": 0.508938512336319, \"nll_celda_med3\": 0.5119339812997068, \"nll_vistas\": [0.508938512336319, 0.5119339812997068, 0.5422007896599855]}, {\"tarea\": \"271d71e2\", \"estrato\": \"barata\", \"celdas\": 1173, \"n_tok\": 2496, \"exacta\": false, \"nll_celda\": 0.13045289022538364, \"nll_celda_med3\": 0.14902079420423062, \"nll_vistas\": [0.13045289022538364, 0.14902079420423062, 0.1606895921581948]}, {\"tarea\": \"db695cfb\", \"estrato\": \"media\", \"celdas\": 1222, \"n_tok\": 2642, \"exacta\": false, \"nll_celda\": 0.2188380065018924, \"nll_celda_med3\": 0.18919067882281865, \"nll_vistas\": [0.2188380065018924, 0.18890313431010894, 0.18919067882281865]}, {\"tarea\": \"7491f3cf\", \"estrato\": \"media\", \"celdas\": 875, \"n_tok\": 1850, \"exacta\": false, \"nll_celda\": 0.10385313197544643, \"nll_celda_med3\": 0.09703413609095982, \"nll_vistas\": [0.10385313197544643, 0.096714111328125, 0.09703413609095982]}, {\"tarea\": \"abc82100\", \"estrato\": \"media\", \"celdas\": 1114, \"n_tok\": 2394, \"exacta\": false, \"nll_celda\": 0.22251672933088812, \"nll_celda_med3\": 0.2274312647711653, \"nll_vistas\": [0.22251672933088812, 0.2274312647711653, 0.2402709741780745]}, {\"tarea\": \"b6f77b65\", \"estrato\": \"media\", \"celdas\": 864, \"n_tok\": 1908, \"exacta\": false, \"nll_celda\": 0.24580291465476709, \"nll_celda_med3\": 0.1748348871866862, \"nll_vistas\": [0.24580291465476709, 0.1748348871866862, 0.13459756639268664]}, {\"tarea\": \"800d221b\", \"estrato\": \"media\", \"celdas\": 1656, \"n_tok\": 3488, \"exacta\": false, \"nll_celda\": 0.14093595311261606, \"nll_celda_med3\": 0.13585078658689048, \"nll_vistas\": [0.14093595311261606, 0.11348079368112168, 0.13585078658689048]}, {\"tarea\": \"8f3a5a89\", \"estrato\": \"media\", \"celdas\": 900, \"n_tok\": 1940, \"exacta\": false, \"nll_celda\": 0.16730750189887153, \"nll_celda_med3\": 0.15820641411675349, \"nll_vistas\": [0.16730750189887153, 0.15218122694227432, 0.15820641411675349]}, {\"tarea\": \"8698868d\", \"estrato\": \"media\", \"celdas\": 622, \"n_tok\": 2856, \"exacta\": false, \"nll_celda\": 0.23121584190049738, \"nll_celda_med3\": 0.24111531248429963, \"nll_vistas\": [0.23121584190049738, 0.25001079399869375, 0.24111531248429963]}, {\"tarea\": \"409aa875\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.13977590203285217, \"nll_celda_med3\": 0.13977590203285217, \"nll_vistas\": [0.13977590203285217, 0.13433215022087097, 0.14551764726638794]}, {\"tarea\": \"d59b0160\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.06821700185537338, \"nll_celda_med3\": 0.06506356596946716, \"nll_vistas\": [0.06821700185537338, 0.06207840517163277, 0.06506356596946716]}, {\"tarea\": \"e376de54\", \"estrato\": \"media\", \"celdas\": 1024, \"n_tok\": 2200, \"exacta\": false, \"nll_celda\": 0.08415575325489044, \"nll_celda_med3\": 0.08415575325489044, \"nll_vistas\": [0.08415575325489044, 0.0789736807346344, 0.08900076150894165]}, {\"tarea\": \"58490d8a\", \"estrato\": \"media\", \"celdas\": 257, \"n_tok\": 2334, \"exacta\": false, \"nll_celda\": 0.21401339282321558, \"nll_celda_med3\": 0.2805024974540978, \"nll_vistas\": [0.21401339282321558, 0.29591535872522495, 0.2805024974540978]}, {\"tarea\": \"6ffbe589\", \"estrato\": \"media\", \"celdas\": 538, \"n_tok\": 2288, \"exacta\": false, \"nll_celda\": 0.4555133975571416, \"nll_celda_med3\": 0.4555133975571416, \"nll_vistas\": [0.4555133975571416, 0.45394256478348627, 0.466349406756433]}]}}")  # brazos q3 medidos en v2 (scripts/_sft35_v2_q3.json)
for arm, seed, p, fh in procs:
    f = "/kaggle/working/%s_s%d/res.json" % (arm, seed)
    if os.path.exists(f): R["%s_s%d" % (arm, seed)] = json.load(open(f))
json.dump(R, open("/kaggle/working/sft35.json", "w"))
for arm, seed, p, fh in procs:
    print("=" * 12 + (" LOG %s_s%d " % (arm, seed)) + "=" * 12, flush=True)
    try: print(open("/kaggle/working/log_%s_s%d.txt" % (arm, seed)).read()[-3000:], flush=True)
    except Exception as e: print("sin log", e, flush=True)

import math
def med(xs):
    xs = sorted(xs); n = len(xs)
    return float("nan") if not n else (xs[n // 2] if n % 2 else 0.5 * (xs[n // 2 - 1] + xs[n // 2]))
def signo_p(k, n):
    return sum(math.comb(n, j) for j in range(k, n + 1)) / 2 ** n if n else float("nan")
def tabla(a, b, campo="nll_celda", fase="eval_post"):
    if a not in R or b not in R: return None
    A = {f["tarea"]: f for f in R[a][fase]}; B = {f["tarea"]: f for f in R[b][fase]}
    out = {}
    for estr in ("barata", "media", "todas"):
        ks = [k for k in A if k in B and campo in A[k] and campo in B[k] and (estr == "todas" or A[k]["estrato"] == estr)]
        if not ks: continue
        ratios = [B[k][campo] / max(1e-6, A[k][campo]) for k in ks]
        gana = sum(1 for r in ratios if r < 1)
        out[estr] = {"n": len(ks), "ratio_med": round(med(ratios), 3), "ratio_media": round(sum(ratios) / len(ks), 3), "gana_b": gana, "p_signo": round(signo_p(gana, len(ks)), 4),
                     "exactas_a": sum(1 for k in ks if A[k].get("exacta")), "exactas_b": sum(1 for k in ks if B[k].get("exacta"))}
    return out
print("\n" + "=" * 70, flush=True)
for tag, r in R.items():
    tr = r.get("train", {})
    pre = [f["nll_celda"] for f in r["eval_pre"] if "nll_celda" in f]; post = [f["nll_celda"] for f in r["eval_post"] if "nll_celda" in f]
    print("%-7s %-28s seq %5d tok %9d %5d tok/s loss_fin %.4f ckpt=%s | NLL/celda med pre %.3f -> post %.3f | exactas pre %d post %d" % (
        tag, r["clase"][:28], tr.get("seq_hechas", 0), tr.get("tok", 0), tr.get("tok_s", 0), (tr.get("curva") or [{}])[-1].get("loss", float("nan")), tr.get("ckpt"),
        med(pre), med(post), sum(1 for f in r["eval_pre"] if f.get("exacta")), sum(1 for f in r["eval_post"] if f.get("exacta"))), flush=True)
print("\nPAREADO post-SFT (ratio = q35 / q3 de NLL/celda; <1 = gana 3.5):", flush=True)
for a, b in (("q3_s1", "q35_s1"), ("q3_s2", "q35_s2"), ("q3_s1", "q35_s3"), ("q3_s2", "q35_s4"), ("q3_s1", "q3_s2"), ("q35_s1", "q35_s2"), ("q35_s3", "q35_s4")):
    print("  %s vs %s:" % (a, b), json.dumps(tabla(a, b)), flush=True)
print("  mediana de 3 vistas, s1:", json.dumps(tabla("q3_s1", "q35_s1", campo="nll_celda_med3")), flush=True)
print("  mediana de 3 vistas, s2:", json.dumps(tabla("q3_s2", "q35_s2", campo="nll_celda_med3")), flush=True)
print("PAREADO pre-SFT (enlace con gate35b):", json.dumps(tabla("q3_s1", "q35_s1", fase="eval_pre")), flush=True)
print("\nPRERREG: la via 3.5 se REABRE si ratio post <= 0,85 en AMBAS semillas y gana >= 2/3 (p<0,05), con la banda s1-vs-s2 del mismo brazo mas estrecha que el efecto.", flush=True)
print("RESULT_JSON:" + json.dumps({"brazos": list(R), "wall_s": round(time.time() - T0)}), flush=True)
